# Blender 5.2.0 LTS — Google Colab GPU Renderer

This notebook renders an existing `.blend` scene with **Cycles on the Colab NVIDIA GPU**.

It is designed for your Fantasy Chess scene workflow:
- build the scene locally with your custom add-on
- save the finished `.blend`
- upload it to Google Drive
- render on Colab instead of your own GPU

### Before running
In Colab, select:

**Runtime → Change runtime type → GPU**

Then use **Runtime → Run all**.


In [ ]:
# ============================================================
# 1) BASIC CONFIGURATION
# Edit this cell for normal use.
# ============================================================

BLENDER_VERSION = "5.2.0"

# Put your .blend here in Google Drive.
BLEND_PATH = "/content/drive/MyDrive/Blender/chess.blend"

# Render output folder in Google Drive.
OUTPUT_DIR = "/content/drive/MyDrive/Blender/renders"

# PNG render mode only: "STILL" or "ANIMATION"
# If OUTPUT_TYPE is VIDEO, animation mode is selected automatically.
MODE = "STILL"

# Single-frame render
FRAME = 1

# Animation range
START_FRAME = 1
END_FRAME = 240

# Leave blank to use the active camera saved in the .blend.
# If your scene contains this camera, you can use:
# CAMERA_NAME = "FantasyChess_Camera"
CAMERA_NAME = ""



# False = always render the full requested sample count.
# True = let Cycles stop clean pixels early.
ADAPTIVE_SAMPLING = False

DENOISE = True

print("Basic configuration loaded.")


In [ ]:
# ============================================================
# 2) QUALITY SELECTOR
# Run this cell to choose your still-render quality.
#
# Change ONLY the next line if you want a different quality:
#     QUALITY_PRESET = "FAST"
#     QUALITY_PRESET = "MEDIUM"
#     QUALITY_PRESET = "HIGH"
#
# Default: MEDIUM
# ============================================================

QUALITY_PRESET = "MEDIUM"

QUALITY_SAMPLES = {
    "FAST": 256,    # quick previews / test renders
    "MEDIUM": 1024, # balanced final quality
    "HIGH": 4096,   # very clean stills, slower
}

QUALITY_PRESET = QUALITY_PRESET.upper().strip()

if QUALITY_PRESET not in QUALITY_SAMPLES:
    raise ValueError(
        f"Unknown QUALITY_PRESET: {QUALITY_PRESET!r}. "
        'Use "FAST", "MEDIUM", or "HIGH".'
    )

SAMPLES = QUALITY_SAMPLES[QUALITY_PRESET]

print("Quality selection loaded.")
print(f"Selected preset: {QUALITY_PRESET}")
print(f"Cycles samples: {SAMPLES}")


In [ ]:
# ============================================================
# 3) OUTPUT TYPE + FILE SETTINGS
# Colab shows controls for lines marked with #@param.
# ============================================================

# PNG is the safer default for long Colab renders.
OUTPUT_TYPE = "PNG" #@param ["PNG", "VIDEO"]

# -------------------- PNG SETTINGS ---------------------------
PNG_COLOR_DEPTH = "16" #@param ["8", "16"]
PNG_COLOR_MODE = "RGB" #@param ["RGB", "RGBA"]
PNG_COMPRESSION = 30 #@param {type:"slider", min:0, max:100, step:5}
UNIQUE_STILL_NAMES = True #@param {type:"boolean"}
SKIP_EXISTING_PNG_FRAMES = True #@param {type:"boolean"}

# ------------------- VIDEO SETTINGS --------------------------
# VIDEO requires MODE = "ANIMATION".
VIDEO_FILENAME = "render" #@param {type:"string"}
VIDEO_FPS = 24 #@param {type:"integer"}
VIDEO_QUALITY = "HIGH" #@param ["LOSSLESS", "PERC_LOSSLESS", "HIGH", "MEDIUM", "LOW"]
VIDEO_ENCODING_SPEED = "GOOD" #@param ["BEST", "GOOD", "REALTIME"]
VIDEO_INCLUDE_AUDIO = False #@param {type:"boolean"}
VIDEO_AUDIO_BITRATE = 192 #@param {type:"integer"}

print('Output type:', OUTPUT_TYPE)
if OUTPUT_TYPE == 'PNG':
    print(f'PNG: {PNG_COLOR_DEPTH}-bit {PNG_COLOR_MODE}, compression={PNG_COMPRESSION}')
else:
    print(f'VIDEO: MP4/H.264, {VIDEO_FPS} fps, quality={VIDEO_QUALITY}, speed={VIDEO_ENCODING_SPEED}, audio={VIDEO_INCLUDE_AUDIO}')


In [ ]:
# ============================================================
# 2) MOUNT GOOGLE DRIVE + VERIFY GPU
# ============================================================

from google.colab import drive
import os
import subprocess

drive.mount("/content/drive")

print("\n--- GPU CHECK ---")
gpu = subprocess.run(
    ["nvidia-smi"],
    text=True,
    capture_output=True
)

if gpu.returncode != 0:
    raise RuntimeError(
        "No NVIDIA GPU is attached. "
        "Choose Runtime > Change runtime type > GPU, "
        "restart the runtime, then use Runtime > Run all."
    )

print(gpu.stdout)

if not os.path.isfile(BLEND_PATH):
    raise FileNotFoundError(
        f"Blend file not found:\n{BLEND_PATH}\n\n"
        "Upload the .blend to Google Drive, then update BLEND_PATH "
        "in the configuration cell."
    )

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Blend file found:", BLEND_PATH)
print("Output folder:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 3) INSTALL BLENDER 5.2.0
# Uses Blender's mirror first to avoid Cloudflare 403 errors.
# ============================================================

import pathlib
import subprocess
import os

# Self-contained fallback, so this cell does not fail with NameError.
BLENDER_VERSION = globals().get("BLENDER_VERSION", "5.2.0")

filename = f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
archive = pathlib.Path(f"/content/{filename}")
install_dir = pathlib.Path(f"/content/blender-{BLENDER_VERSION}-linux-x64")
blender_bin = install_dir / "blender"

# Small runtime libraries Blender may need in Colab.
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(
    [
        "apt-get", "install", "-y", "-qq",
        "libxi6", "libxxf86vm1", "libxfixes3",
        "libxrender1", "libgl1", "libsm6", "libegl1",
        "curl"
    ],
    check=True
)

urls = [
    f"https://mirror.blender.org/release/Blender5.2/{filename}",
    f"https://ftp.nluug.nl/pub/graphics/blender/release/Blender5.2/{filename}",
    f"https://download.blender.org/release/Blender5.2/{filename}",
]

if not blender_bin.exists():
    if archive.exists():
        archive.unlink()

    success = False

    for url in urls:
        print("\nTrying:", url)

        result = subprocess.run(
            [
                "curl",
                "-fL",
                "--retry", "3",
                "--retry-delay", "2",
                "--connect-timeout", "20",
                "-A", "Mozilla/5.0",
                "-o", str(archive),
                url
            ]
        )

        if (
            result.returncode == 0
            and archive.exists()
            and archive.stat().st_size > 300_000_000
        ):
            print(
                f"Downloaded {archive.stat().st_size / (1024**2):.1f} MiB"
            )
            success = True
            break

        if archive.exists():
            archive.unlink()

    if not success:
        raise RuntimeError(
            "Blender could not be downloaded from any configured mirror."
        )

    print("\nExtracting Blender...")
    subprocess.run(
        ["tar", "-xf", str(archive), "-C", "/content"],
        check=True
    )

if not blender_bin.exists():
    raise FileNotFoundError(
        f"Blender binary not found after extraction: {blender_bin}"
    )

print("\n--- BLENDER VERSION ---")
subprocess.run([str(blender_bin), "--version"], check=True)

BLENDER_BIN = str(blender_bin)
print("BLENDER_BIN =", BLENDER_BIN)


In [ ]:
driver_code = 'import bpy\nimport os\nimport sys\nimport argparse\n\ndef cli_args():\n    if "--" not in sys.argv:\n        return []\n    return sys.argv[sys.argv.index("--") + 1:]\n\np = argparse.ArgumentParser()\np.add_argument("--output-dir", required=True)\np.add_argument("--mode", choices=["STILL","ANIMATION"], default="STILL")\np.add_argument("--frame", type=int, default=1)\np.add_argument("--start-frame", type=int, default=1)\np.add_argument("--end-frame", type=int, default=240)\np.add_argument("--samples", type=int, default=1024)\np.add_argument("--adaptive", action="store_true")\np.add_argument("--denoise", action="store_true")\np.add_argument("--camera", default="")\np.add_argument("--output-type", choices=["PNG","VIDEO"], default="PNG")\np.add_argument("--png-depth", choices=["8","16"], default="16")\np.add_argument("--png-color-mode", choices=["RGB","RGBA"], default="RGB")\np.add_argument("--png-compression", type=int, default=30)\np.add_argument("--unique-still-names", action="store_true")\np.add_argument("--skip-existing-png-frames", action="store_true")\np.add_argument("--video-filename", default="render")\np.add_argument("--video-fps", type=int, default=24)\np.add_argument("--video-quality", choices=["LOSSLESS","PERC_LOSSLESS","HIGH","MEDIUM","LOW"], default="HIGH")\np.add_argument("--video-encoding-speed", choices=["BEST","GOOD","REALTIME"], default="GOOD")\np.add_argument("--video-audio", action="store_true")\np.add_argument("--video-audio-bitrate", type=int, default=192)\na = p.parse_args(cli_args())\n\nscene = bpy.context.scene\nos.makedirs(a.output_dir, exist_ok=True)\n\n# Camera\nif a.camera:\n    cam=bpy.data.objects.get(a.camera)\n    if cam is None or cam.type != "CAMERA": raise RuntimeError(f"Camera {a.camera!r} not found")\n    scene.camera=cam\nif scene.camera is None:\n    preferred=bpy.data.objects.get("FantasyChess_Camera")\n    if preferred and preferred.type=="CAMERA": scene.camera=preferred\n    else:\n        cams=[o for o in bpy.data.objects if o.type=="CAMERA"]\n        if not cams: raise RuntimeError("No camera exists in this .blend file")\n        scene.camera=cams[0]\nprint(\'Using camera:\', scene.camera.name)\n\n# Cycles GPU\nscene.render.engine=\'CYCLES\'\nscene.cycles.device=\'GPU\'\ncycles=bpy.context.preferences.addons.get(\'cycles\')\nif cycles is None: raise RuntimeError(\'Cycles add-on unavailable\')\nprefs=cycles.preferences\nbackend=None\nfor candidate in (\'OPTIX\',\'CUDA\'):\n    try:\n        prefs.compute_device_type=candidate\n        prefs.get_devices()\n        devices=list(prefs.devices)\n        gpus=[d for d in devices if d.type != \'CPU\']\n        if gpus:\n            for d in devices: d.use=(d in gpus)\n            backend=candidate\n            break\n    except Exception as e:\n        print(candidate, \'unavailable:\', e)\nif backend is None: raise RuntimeError(\'No usable NVIDIA GPU detected\')\nprint(\'Cycles backend:\', backend)\n\nscene.cycles.samples=max(1,a.samples)\nscene.cycles.use_adaptive_sampling=bool(a.adaptive)\nscene.cycles.use_denoising=bool(a.denoise)\nfor name,value in {\'max_bounces\':16,\'diffuse_bounces\':6,\'glossy_bounces\':8,\'transmission_bounces\':12,\'transparent_max_bounces\':12}.items():\n    if hasattr(scene.cycles,name): setattr(scene.cycles,name,value)\nif hasattr(scene.render,\'use_persistent_data\'): scene.render.use_persistent_data=(a.mode==\'ANIMATION\')\n\ndef unique_path(base, ext):\n    first=base+ext\n    if not os.path.exists(first): return first\n    n=2\n    while True:\n        candidate=f"{base}_v{n}{ext}"\n        if not os.path.exists(candidate): return candidate\n        n+=1\n\ndef configure_png():\n    im=scene.render.image_settings\n    im.file_format=\'PNG\'\n    im.color_depth=a.png_depth\n    im.color_mode=a.png_color_mode\n    im.compression=max(0,min(100,a.png_compression))\n    scene.render.use_file_extension=True\n    print(f\'PNG settings: {im.color_depth}-bit {im.color_mode}, compression={im.compression}\')\n\ndef render_png_still(frame):\n    configure_png()\n    base=os.path.join(a.output_dir,f\'frame_{frame:04d}\')\n    final=unique_path(base,\'.png\') if a.unique_still_names else base+\'.png\'\n    scene.frame_set(frame)\n    scene.render.filepath=os.path.splitext(final)[0]\n    print(\'Rendering still ->\', final)\n    bpy.ops.render.render(write_still=True)\n    if not os.path.exists(final): raise RuntimeError(f\'Expected PNG not created: {final}\')\n\ndef render_png_animation():\n    configure_png()\n    if a.end_frame<a.start_frame: raise RuntimeError(\'END_FRAME must be >= START_FRAME\')\n    for frame in range(a.start_frame,a.end_frame+1):\n        final=os.path.join(a.output_dir,f\'frame_{frame:04d}.png\')\n        if a.skip_existing_png_frames and os.path.exists(final) and os.path.getsize(final)>0:\n            print(\'SKIP\',frame,\'->\',final); continue\n        scene.frame_set(frame)\n        scene.render.filepath=os.path.join(a.output_dir,f\'frame_{frame:04d}\')\n        print(\'RENDER\',frame,\'->\',final)\n        bpy.ops.render.render(write_still=True)\n        if not os.path.exists(final): raise RuntimeError(f\'Expected PNG not created: {final}\')\n\ndef render_video():\n    if a.mode!=\'ANIMATION\': raise RuntimeError(\'VIDEO output requires MODE="ANIMATION"\')\n    if a.end_frame<a.start_frame: raise RuntimeError(\'END_FRAME must be >= START_FRAME\')\n    scene.frame_start=a.start_frame\n    scene.frame_end=a.end_frame\n    scene.render.image_settings.file_format=\'FFMPEG\'\n    ff=scene.render.ffmpeg\n    ff.format=\'MPEG4\'\n    ff.codec=\'H264\'\n    ff.constant_rate_factor=a.video_quality\n    ff.ffmpeg_preset=a.video_encoding_speed\n    if a.video_audio:\n        ff.audio_codec=\'AAC\'\n        ff.audio_bitrate=max(32,min(384,a.video_audio_bitrate))\n        try: ff.audio_channels=\'STEREO\'\n        except Exception: pass\n    else:\n        ff.audio_codec=\'NONE\'\n    scene.render.fps=max(1,a.video_fps)\n    scene.render.fps_base=1.0\n    scene.render.use_file_extension=True\n    name=(a.video_filename.strip() or \'render\')\n    final=unique_path(os.path.join(a.output_dir,name),\'.mp4\')\n    scene.render.filepath=os.path.splitext(final)[0]\n    print(\'Video: MPEG4/H264\', scene.render.fps, \'fps, quality=\',ff.constant_rate_factor,\'speed=\',ff.ffmpeg_preset,\'audio=\',ff.audio_codec)\n    print(\'Rendering video ->\', final)\n    bpy.ops.render.render(animation=True)\n    if not os.path.exists(final):\n        near=[f for f in os.listdir(a.output_dir) if f.startswith(os.path.splitext(os.path.basename(final))[0])]\n        raise RuntimeError(f\'Expected video not found: {final}; nearby={near}\')\n\nprint(\'Samples:\',scene.cycles.samples,\'Resolution:\',scene.render.resolution_x,\'x\',scene.render.resolution_y)\nif a.output_type==\'VIDEO\': render_video()\nelif a.mode==\'STILL\': render_png_still(a.frame)\nelse: render_png_animation()\nprint(\'Render complete.\')\n'
DRIVER_PATH = "/content/blender_colab_render_driver.py"

with open(DRIVER_PATH, "w", encoding="utf-8") as f:
    f.write(driver_code)

print("Render driver created:", DRIVER_PATH)


In [ ]:

# ============================================================
# 6) RENDER
# This cell is self-contained with respect to mode selection.
# ============================================================

import os
import subprocess

required = [
    "BLEND_PATH", "OUTPUT_DIR", "MODE", "FRAME",
    "START_FRAME", "END_FRAME", "SAMPLES",
    "ADAPTIVE_SAMPLING", "DENOISE", "CAMERA_NAME",
    "BLENDER_BIN", "DRIVER_PATH",
    "OUTPUT_TYPE",
    "PNG_COLOR_DEPTH", "PNG_COLOR_MODE", "PNG_COMPRESSION",
    "UNIQUE_STILL_NAMES", "SKIP_EXISTING_PNG_FRAMES",
    "VIDEO_FILENAME", "VIDEO_FPS", "VIDEO_QUALITY",
    "VIDEO_ENCODING_SPEED", "VIDEO_INCLUDE_AUDIO",
    "VIDEO_AUDIO_BITRATE",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Required settings are missing: "
        + ", ".join(missing)
        + ". Use Runtime > Run all."
    )

if not os.path.isfile(BLEND_PATH):
    raise FileNotFoundError(BLEND_PATH)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# IMPORTANT:
# VIDEO always renders START_FRAME through END_FRAME automatically.
# MODE is only used for PNG output.
EFFECTIVE_MODE = "ANIMATION" if OUTPUT_TYPE == "VIDEO" else MODE

print("Starting Blender render...")
print("Output type:", OUTPUT_TYPE)
print("Effective render mode:", EFFECTIVE_MODE)
print("Samples:", SAMPLES)

if OUTPUT_TYPE == "VIDEO":
    print(f"Video frame range: {START_FRAME} to {END_FRAME}")
else:
    if EFFECTIVE_MODE == "STILL":
        print(f"PNG still frame: {FRAME}")
    else:
        print(f"PNG animation frame range: {START_FRAME} to {END_FRAME}")

cmd = [
    BLENDER_BIN,
    "-b", BLEND_PATH,
    "--python", DRIVER_PATH,
    "--",
    "--output-dir", OUTPUT_DIR,
    "--mode", EFFECTIVE_MODE,
    "--frame", str(FRAME),
    "--start-frame", str(START_FRAME),
    "--end-frame", str(END_FRAME),
    "--samples", str(SAMPLES),
    "--output-type", OUTPUT_TYPE,

    "--png-depth", str(PNG_COLOR_DEPTH),
    "--png-color-mode", PNG_COLOR_MODE,
    "--png-compression", str(PNG_COMPRESSION),

    "--video-filename", VIDEO_FILENAME,
    "--video-fps", str(VIDEO_FPS),
    "--video-quality", VIDEO_QUALITY,
    "--video-encoding-speed", VIDEO_ENCODING_SPEED,
    "--video-audio-bitrate", str(VIDEO_AUDIO_BITRATE),
]

if ADAPTIVE_SAMPLING:
    cmd.append("--adaptive")

if DENOISE:
    cmd.append("--denoise")

if CAMERA_NAME:
    cmd += ["--camera", CAMERA_NAME]

if UNIQUE_STILL_NAMES:
    cmd.append("--unique-still-names")

if SKIP_EXISTING_PNG_FRAMES:
    cmd.append("--skip-existing-png-frames")

if VIDEO_INCLUDE_AUDIO:
    cmd.append("--video-audio")

print("\nLaunching Blender...\n")
subprocess.run(cmd, check=True)


## Quality selector

The notebook now has a dedicated **Quality Selector** cell near the top.

Just run that cell as-is for the default:

```python
QUALITY_PRESET = "MEDIUM"
```

Or change it to one of these:

```python
QUALITY_PRESET = "FAST"
QUALITY_PRESET = "MEDIUM"
QUALITY_PRESET = "HIGH"
```

Preset summary:

| Preset | Samples | Intended use |
|---|---:|---|
| `FAST` | 256 | Quick previews and test stills |
| `MEDIUM` | 1024 | Balanced final still quality |
| `HIGH` | 4096 | Very clean final stills, slower |

The rest of the notebook automatically uses the selected sample count.

Denoising remains enabled by default, and adaptive sampling remains off by default so Cycles renders the full selected sample count.


## Still output naming behavior

This version adds:

```python
UNIQUE_STILL_NAMES = True
```

When you rerun a **still** render and the output already exists, the notebook creates a new filename instead of skipping or overwriting:

- `frame_0001.png`
- `frame_0001_v2.png`
- `frame_0001_v3.png`

Animation mode is unchanged: existing animation frames are still skipped so interrupted renders can resume safely.


## PNG vs Video

Choose the output in the **Output Type + File Settings** cell.

- `OUTPUT_TYPE = "PNG"`: saves stills or an animation image sequence. PNG animation frames can resume after an interrupted Colab session.
- `OUTPUT_TYPE = "VIDEO"`: requires `MODE = "ANIMATION"` and writes an MP4/H.264 video. You can set the filename, FPS, quality, encoding speed, and audio options.
- Existing stills and videos get `_v2`, `_v3`, etc. instead of being overwritten.

For long renders, PNG sequence mode is safer because a direct video render cannot resume cleanly after a disconnect.


## Automatic video mode

When you select:

```python
OUTPUT_TYPE = "VIDEO"
```

the notebook automatically renders `START_FRAME` through `END_FRAME` as an animation.

You no longer need to change `MODE` to `"ANIMATION"` for video output.

`MODE` is only used when `OUTPUT_TYPE = "PNG"`:
- `MODE = "STILL"` → one PNG
- `MODE = "ANIMATION"` → PNG frame sequence


## Render-mode fix

The render cell now defines:

```python
EFFECTIVE_MODE = "ANIMATION" if OUTPUT_TYPE == "VIDEO" else MODE
```

inside the render cell itself.

This means:
- `OUTPUT_TYPE = "VIDEO"` automatically renders the animation range.
- `OUTPUT_TYPE = "PNG"` uses the selected `MODE`.
- Restarting the Colab runtime will not leave `EFFECTIVE_MODE` undefined.
